# 07 - Sub-experiment 3: Few-shot vs Zero-shot

This notebook compares plain zero-shot against a few-shot prefixed variant using the best prompt labels selected in Sub-experiment 2.

In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
import sys
sys.modules["tensorflow"] = None

### What this does and why

We test whether adding short example context before each post improves BART MNLI zero-shot performance compared with no examples.

In [2]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns

root = Path.cwd()
outputs_dir = root / "outputs"
if not outputs_dir.exists():
    outputs_dir = root.parent / "outputs"

sample_df = pd.read_csv(outputs_dir / "llm_sample.csv")
best_prompt = pd.read_json(outputs_dir / "best_prompt.json", typ="series")
best_labels = [best_prompt["label_0_text"], best_prompt["label_1_text"]]

texts = sample_df["text_clean"].astype(str).tolist()
y_true = sample_df["risk_label"].astype(int).to_numpy()

def compute_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {
        "accuracy": float(acc),
        "f1_weighted": float(f1_w),
        "f1_macro": float(f1_m),
    }

def run_inference(classifier, text_list, labels, desc):
    preds, confs = [], []
    for t in tqdm(text_list, desc=desc):
        out = classifier(t, candidate_labels=labels)
        winner = out["labels"][0]
        score = float(out["scores"][0])
        pred = 0 if winner == labels[0] else 1
        preds.append(pred)
        confs.append(score)
    return np.array(preds, dtype=int), np.array(confs, dtype=float)

few_shot_prefix = (
    "Example 1 (Mental Health Risk): \"i've been feeling really down lately and can't seem to find any motivation to do anything\"\n"
    "Example 2 (Mental Health Risk): \"the anxiety is getting worse and i don't know how to cope anymore\"\n"
    "Example 3 (High Risk Suicidal): \"i've been thinking about ending it all and have a plan ready for tonight\"\n"
    "Example 4 (High Risk Suicidal): \"i said goodbye to everyone today nobody knows why but i do\"\n"
    "Now classify the following post:\n\n"
)

try:
    classifier = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli",
        device=-1,
    )
except Exception as e:
    raise RuntimeError(
        "Failed to load facebook/bart-large-mnli. Install transformers and torch. "
        f"Original error: {e}"
    )

zs_pred, zs_conf = run_inference(classifier, texts, best_labels, "Zero-shot inference")
few_texts = [few_shot_prefix + t for t in texts]
fs_pred, fs_conf = run_inference(classifier, few_texts, best_labels, "Few-shot inference")

zs_m = compute_metrics(y_true, zs_pred)
fs_m = compute_metrics(y_true, fs_pred)

result = pd.DataFrame([
    {"Method": "Zero-Shot", "Accuracy": zs_m["accuracy"], "Weighted F1": zs_m["f1_weighted"], "Macro F1": zs_m["f1_macro"]},
    {"Method": "Few-Shot", "Accuracy": fs_m["accuracy"], "Weighted F1": fs_m["f1_weighted"], "Macro F1": fs_m["f1_macro"]},
])
result.to_csv(outputs_dir / "fewshot_vs_zeroshot.csv", index=False)

plt.figure(figsize=(5, 4))
sns.barplot(data=result, x="Method", y="Weighted F1")
plt.ylim(0, 1)
plt.title("Few-shot vs Zero-shot (Weighted F1)")
plt.tight_layout()
plt.savefig(outputs_dir / "fewshot_vs_zeroshot_f1.png", dpi=200)
plt.close()

winner = "Few-Shot" if fs_m["f1_weighted"] > zs_m["f1_weighted"] else "Zero-Shot"
delta = float(fs_m["f1_weighted"] - zs_m["f1_weighted"])
meta = {
    "winner": winner,
    "f1_delta_few_minus_zero": delta,
    "best_labels": best_labels,
}
(outputs_dir / "fewshot_vs_zeroshot_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

preds = sample_df[["row_id", "text_clean", "risk_label"]].copy()
preds["zero_shot_pred"] = zs_pred
preds["zero_shot_confidence"] = zs_conf
preds["few_shot_pred"] = fs_pred
preds["few_shot_confidence"] = fs_conf
preds.to_csv(outputs_dir / "fewshot_vs_zeroshot_predictions.csv", index=False)

print("Winner:", winner)
print("Weighted F1 difference (Few - Zero):", delta)
result

C:\Users\HP\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Few-shot inference: 100%|██████████| 200/200 [13:57<00:00,  4.19s/it]


Winner: Zero-Shot
Weighted F1 difference (Few - Zero): -0.10055925193021487


,Method,Accuracy,Weighted F1,Macro F1
0,Zero-Shot,0.710,0.703325,0.703325
1,Few-Shot,0.605,0.602766,0.602766
